In [0]:
%sql

INSERT OVERWRITE proyecto_final.silver.kioscos (
    nombre,
    barrio,
    comuna,
    direccion,
    telefono,
    sitio_web,
    categoria,
    email,
    instagram,
    facebook,
    rating,
    reviews_count,
    claimed,
    top_review,
    horario,
    latitud,
    longitud,
    tipo_entidad,
    coordenadas
)
WITH datos_limpios AS (
    SELECT 
        -- Limpieza de texto y caracteres raros
        LOWER(REPLACE(REPLACE(TRIM(name), '"', ''), '', '')) AS nombre,
        LOWER(REPLACE(REPLACE(TRIM(category), '"', ''), '', '')) AS categoria,
        LOWER(REPLACE(REPLACE(TRIM(address), '"', ''), '', '')) AS direccion,
        
        -- Datos de contacto
        REPLACE(REPLACE(REPLACE(TRIM(phone), '"', ''), '', ''), '', '') AS telefono,
        REPLACE(REPLACE(TRIM(website), '"', ''), '', '') AS sitio_web,
        LOWER(REPLACE(REPLACE(TRIM(email), '"', ''), '', '')) AS email,
        LOWER(REPLACE(REPLACE(TRIM(instagram), '"', ''), '', '')) AS instagram,
        LOWER(REPLACE(REPLACE(TRIM(facebook), '"', ''), '', '')) AS facebook,
        
        -- Metadatos y reseñas
        rating,
        reviews_count,
        CAST(claimed AS BOOLEAN) AS claimed,
        LOWER(REPLACE(REPLACE(TRIM(top_Review), '"', ''), '', '')) AS top_review,
        LOWER(REPLACE(REPLACE(TRIM(working_Hours), '"', ''), '', '')) AS horario,
        
        -- Geografía y estandarización
        latitude AS latitud,
        longitude AS longitud,
        'kiosco' AS tipo_entidad,
        CONCAT('POINT (', longitude, ' ', latitude, ')') AS coordenadas,
        fecha_ingesta
    FROM proyecto_final.raw.kioscos_bronze
    WHERE 
        name IS NOT NULL AND latitude IS NOT NULL 
      AND longitude IS NOT NULL
      AND LOWER(REPLACE(REPLACE(TRIM(category), '"', ''), '', '')) IN ('kiosco', 'quiosco', 'tienda de golosinas', 'comercio','almacén')
),
datos_deduplicados AS (
    SELECT *,
           -- Deduplicación particionando por el nombre ya limpio y sus coordenadas exactas
           ROW_NUMBER() OVER(
               PARTITION BY nombre, latitud, longitud 
               ORDER BY fecha_ingesta DESC
           ) AS rn
    FROM datos_limpios
)
SELECT 
    k.nombre,
    -- barrio y comuna obtenidos por coordenadas
    b.barrio_nombre AS barrio,
    b.comuna_id AS comuna,
    k.direccion,
    k.telefono,
    k.sitio_web,
    CASE WHEN k.categoria= 'kiosco' OR k.categoria= 'quiosco' THEN 'kiosco' ELSE k.categoria END AS categoria,
    k.email,
    k.instagram,
    k.facebook,
    k.rating,
    k.reviews_count,
    k.claimed,
    k.top_review,
    k.horario,
    k.latitud,
    k.longitud,
    k.tipo_entidad,
    k.coordenadas
FROM datos_deduplicados k
LEFT JOIN proyecto_final.silver.barrios b 
  ON st_contains(st_geomfromtext(b.coordenadas), st_point(k.longitud, k.latitud))
WHERE k.rn = 1 AND b.barrio_nombre IS NOT NULL;